In [1]:
# Repo paths 
from pathlib import Path

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / 'data').exists():
            return p
    return start

ROOT = find_repo_root()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PLOTS = ROOT / 'plots'
PLOTS.mkdir(exist_ok=True)

# Imports

In [2]:
import pandas as pd
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

path = ROOT / 'chatlogs.csv'
df = pd.read_csv(path)

print(df.head())

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


   Unnamed: 0              message association_to_offender      time  \
0           0           gold 2 zed                   enemy  00:00:21   
1           1                 IIII                   enemy  00:00:27   
2           2  nice premade lie :o                   enemy  00:00:27   
3           3                  ISI                   enemy  00:00:28   
4           4        smiteless pls                   enemy  00:00:43   

   case_total_reports  allied_report_count  enemy_report_count  \
0                   8                    0                   2   
1                   8                    0                   2   
2                   8                    0                   2   
3                   8                    0                   2   
4                   8                    0                   2   

  most_common_report_reason  chatlog_id champion_name  
0         Negative Attitude           1          Udyr  
1         Negative Attitude           1         Riven  
2 

# Text Cleaning


In [3]:
df['text']=df['message']

In [4]:
df.head()

,Unnamed: 0,message,association_to_offender,time,case_total_reports,allied_report_count,enemy_report_count,most_common_report_reason,chatlog_id,champion_name,text
0,0,gold 2 zed,enemy,00:00:21,8,0,2,Negative Attitude,1,Udyr,gold 2 zed
1,1,IIII,enemy,00:00:27,8,0,2,Negative Attitude,1,Riven,IIII
2,2,nice premade lie :o,enemy,00:00:27,8,0,2,Negative Attitude,1,Udyr,nice premade lie :o
3,3,ISI,enemy,00:00:28,8,0,2,Negative Attitude,1,Riven,ISI
4,4,smiteless pls,enemy,00:00:43,8,0,2,Negative Attitude,1,Udyr,smiteless pls


# Tokenization & Padding

In [ ]:
# 1. Handle missing values: Replace NaNs with empty strings to prevent errors during text concatenation
df['message'] = df['message'].fillna('')

# 2. Group data by chat log and champion
# Concatenate messages for each chat log and champion, separated by a space.
# 'first' retains one association; this grouping still needs validation for mixed speakers.
grouped_df = df.groupby(['chatlog_id', 'champion_name']).agg({
    'message': lambda x: ' '.join(x),
    'association_to_offender': 'first'
}).reset_index()

# 3. Record reported-player status as metadata, not a toxicity target
# This flag describes the retained association, not whether the messages are toxic.
grouped_df['is_reported_player'] = (grouped_df['association_to_offender'] == 'offender').astype(int)

# 4. Display results to verify the new structure
print("Shape of the new aggregated dataset:", grouped_df.shape)
print("\nSample rows:")
print(grouped_df[['chatlog_id', 'champion_name', 'message', 'is_reported_player']].head())

# Inspect reported-player status counts (not toxicity labels)
print("\nReported-player status distribution:")
print(grouped_df['is_reported_player'].value_counts())

In [ ]:



def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'[^a-z0-9\s]', '', text) # deleting chars that are not letters or numbers combination
    text = re.sub(r'\s+', ' ', text).strip() # deleting double spaces

    return text

# applying the clean_text function on the grouped messages
grouped_df['cleaned_text'] = grouped_df['message'].apply(clean_text)

MAX_WORDS = 10000  # vocabulary of 10,000 words
MAX_LEN = 300      # max of 300 words per game (increased from 50 because messages are grouped)

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(grouped_df['cleaned_text'])

sequences = tokenizer.texts_to_sequences(grouped_df['cleaned_text'])

# sentences < 300 words will get padding of zeros to fit 300 words and sentences > 300 words will be cut
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post') 

# Toxicity targets require content-based annotations.
# Clear any target left in memory by a previous run of this cell.
y = None

print("Shape of data tensor (X):", X.shape)
print("Toxicity targets (y) are not available yet; annotate messages before training.")

## Manual annotation pilot

Goal: label the content of individual messages, independent of reported-player status.

Run the first repository-path cell, then the cells in this section. This section does not require TensorFlow or `grouped_df`.

Labels:
- `0`: neutral conversation, teamwork, encouragement, or criticism without personal attacks.
- `1`: insults, harassment, threats, or hateful attacks directed at others.
- `unsure`: unclear meaning, unfamiliar language, or insufficient context. Review later; do not turn this into `0`.

A swear word alone does not automatically establish toxicity. Use context when necessary, but label the target message, not its neighbors. The neighbor columns show adjacent entries from the same chat log in CSV order and may be written by different speakers.

Start with the first 10 messages. This 200-message pilot is for refining annotation rules; it is not a final training or test set. The original message text is preserved. Reported-player roles are deliberately excluded from the annotation view.


In [ ]:
import pandas as pd

SAMPLE_SIZE = 200
RANDOM_STATE = 42

# Read individual messages directly; do not use the champion-level aggregation.
annotation_source = pd.read_csv(
    ROOT / 'chatlogs.csv',
    usecols=['chatlog_id', 'time', 'message'],
    keep_default_na=False,
    dtype={'message': 'string'},
)
# Zero-based row position in the source CSV (excluding its header).
# Keep this source CSV unchanged while annotating this sample.
annotation_source.insert(0, 'message_id', annotation_source.index)

messages_by_log = annotation_source.groupby('chatlog_id', sort=False)['message']
annotation_source['previous_message'] = messages_by_log.shift(1).fillna('')
annotation_source['next_message'] = messages_by_log.shift(-1).fillna('')

has_text = annotation_source['message'].str.strip().ne('')
has_log = annotation_source['chatlog_id'].astype(str).str.strip().ne('')
eligible_messages = annotation_source.loc[has_text & has_log]

annotation_sample = eligible_messages.sample(
    n=min(SAMPLE_SIZE, len(eligible_messages)),
    random_state=RANDOM_STATE,
).copy().reset_index(drop=True)
annotation_sample['toxicity_label'] = pd.Series(
    pd.NA, index=annotation_sample.index, dtype='string'
)

annotation_columns = [
    'message_id', 'chatlog_id', 'time', 'previous_message',
    'message', 'next_message', 'toxicity_label',
]
print(f'Sampled {len(annotation_sample)} messages. No toxicity labels were inferred.')
with pd.option_context('display.max_colwidth', None):
    display(annotation_sample[annotation_columns].head(10))


### Enter the first 10 labels

Read the `message` column and use its `message_id` when entering a label below. Do not use the table's display index.

Add entries inside `manual_labels`, for example `12345: "0",` only if that ID is in your sample and your review supports that label. Allowed values are `"0"`, `"1"`, and `"unsure"`. Leave messages you have not reviewed out of the dictionary.

Save this notebook after editing the dictionary. Rerunning the sampling cell resets the in-memory labels; rerun the dictionary cell to restore your saved labels. Keep `RANDOM_STATE` and the source CSV unchanged during annotation.

The dictionary is empty intentionally. Missing labels remain missing, and `unsure` is reserved for review. No training targets are created here.


In [ ]:
# Enter your own content-based decisions here, then save the notebook.
manual_labels = {
    # message_id: "0", "1", or "unsure"
}

allowed_labels = {'0', '1', 'unsure'}
unknown_ids = set(manual_labels) - set(annotation_sample['message_id'])
invalid_labels = set(manual_labels.values()) - allowed_labels
if unknown_ids:
    raise ValueError(f'Message IDs not present in this sample: {unknown_ids}')
if invalid_labels:
    raise ValueError(f'Use only "0", "1", or "unsure": {invalid_labels}')

annotation_sample['toxicity_label'] = (
    annotation_sample['message_id'].map(manual_labels).astype('string')
)

print('Annotation counts:')
print(annotation_sample['toxicity_label'].fillna('unlabeled').value_counts())
with pd.option_context('display.max_colwidth', None):
    display(annotation_sample[annotation_columns].head(10))
